---
title: 'Lab 6: Optuna'
subtitle: Biblioteki Python w analizie danych
author: Tomasz Rodak
jupyter: python3
---


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rodakt/BPwAD/blob/v2/laby/lab_6.ipynb)

Na wykładzie 3 poznaliśmy Optunę: koncepcję study i trial, metody `suggest_*`, samplery (TPE, Random, Grid) oraz pruning. W tym arkuszu przećwiczymy te narzędzia na dwóch rodzajach problemów — najpierw na optymalizacji funkcji matematycznej (gdzie możemy zobaczyć, co robi sampler na mapie konturowej), a następnie na optymalizacji hiperparametrów modeli uczenia maszynowego z kroswalidacją i pruningiem.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

## 1. Funkcja Himmelblau'a

Funkcja Himmelblau'a to klasyczny benchmark w optymalizacji. Jest zdefiniowana jako:

$$f(x, y) = (x^2 + y - 11)^2 + (x + y^2 - 7)^2$$

Ma cztery minima globalne, w których $f = 0$:

| $x$ | $y$ |
|---|---|
| $3.0$ | $2.0$ |
| $-2.805118$ | $3.131312$ |
| $-3.779310$ | $-3.283186$ |
| $3.584428$ | $-1.848126$ |

Będziemy szukać tych minimów za pomocą Optuny, obserwując, jak różne samplery eksplorują przestrzeń przeszukiwania.

### 1.1 Mapa konturowa

Zanim uruchomimy optymalizację, narysujmy mapę konturową funkcji Himmelblau'a. Będzie nam służyć jako tło do wizualizacji triali.

Utwórz siatkę punktów na $[-5, 5]^2$ za pomocą `np.meshgrid` i oblicz wartości $f(x, y)$ na tej siatce. Narysuj mapę konturową (`plt.contourf`) z nałożonymi liniami konturu (`plt.contour`). Użyj skali logarytmicznej dla poziomów, np. `levels = np.logspace(-2, 3, 30)`, aby lepiej rozróżnić okolicę minimów.

Zaznacz cztery minima globalne kolorowymi punktami.

### 1.2 Funkcja celu

Napisz funkcję `objective(trial)`, która:

1. Pobiera dwie zmienne `x` i `y` z przedziału $[-5, 5]$ za pomocą `trial.suggest_float`.
2. Zwraca wartość funkcji Himmelblau'a.

### 1.3 Optymalizacja z TPESampler

Utwórz study z domyślnym samplerem (`TPESampler`) i uruchom optymalizację z `n_trials=200`. Ustaw `seed` w `TPESampler(seed=...)` dla powtarzalności wyników. Wyświetl najlepszy znaleziony punkt i wartość funkcji celu.

### 1.4 Wizualizacja triali

Pobierz historię triali z `study.trials_dataframe()`. Nałóż punkty na mapę konturową z sekcji 1.1, kolorując je wartością funkcji celu (użyj `plt.scatter` z argumentem `c`). 

Czy TPE skupia próby w okolicy minimów, czy rozrzuca równomiernie?

### 1.5 Powtórzenie z innym seedem

Uruchom optymalizację ponownie z innym `seed` w `TPESampler(seed=...)`. Czy algorytm znalazł inne minimum? Czy rozkład triali wygląda podobnie?

## 2. Porównanie samplerów

Porównamy trzy samplery na tym samym problemie: `TPESampler`, `RandomSampler` i `GridSampler`.

### 2.1 Trzy study

Utwórz trzy study, każde z innym samplerem:

1. **TPE**: `optuna.samplers.TPESampler(seed=42)`, `n_trials=200`.
2. **Random**: `optuna.samplers.RandomSampler(seed=42)`, `n_trials=200`.
3. **Grid**: `optuna.samplers.GridSampler(search_space)`, gdzie `search_space` to słownik z siatkami wartości dla `x` i `y`. Dobierz siatkę tak, aby pokrywała $[-5, 5]^2$ — np. `np.linspace(-5, 5, 15)` dla obu zmiennych (225 punktów).

### 2.2 Porównanie wizualne

Narysuj trzy panele obok siebie (`plt.subplots(1, 3, ...)`), każdy z mapą konturową funkcji Himmelblau'a i nałożonymi punktami triali z odpowiedniego study.

Porównaj rozkłady punktów:

- Czy TPE skupia próby w okolicach minimów?
- Czy Random rozrzuca je równomiernie?
- Jak wygląda regularna siatka Grida?

### 2.3 Zbieżność

Dla każdego study narysuj na jednym wykresie krzywą *best-so-far*: dla każdej próby $t$ wyznacz najlepszą (najmniejszą) wartość funkcji celu wśród prób $1, \ldots, t$.

Który sampler osiągnął dobre rozwiązanie najszybciej? Który potrzebował najwięcej prób?

*Wskazówka:* `np.minimum.accumulate` oblicza bieżące minimum kumulatywne.

## 3. Optymalizacja hiperparametrów z kroswalidacją

Przechodzimy od funkcji matematycznych do rzeczywistego problemu ML. Użyjemy Optuny do optymalizacji hiperparametrów klasyfikatorów na zbiorze danych, korzystając z potoków scikit-learn i kroswalidacji.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

data = load_breast_cancer()
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Treningowy: {X_train.shape}, testowy: {X_test.shape}")

### 3.1 Funkcja celu z warunkową przestrzenią

Napisz funkcję `objective(trial)`, która:

1. Wybiera klasyfikator za pomocą `trial.suggest_categorical("classifier", ["knn", "svc", "logreg"])`.
2. W zależności od wyboru definiuje hiperparametry warunkowe:
   - **knn**: `n_neighbors` (int, 1–30), `weights` (categorical: `"uniform"`, `"distance"`).
   - **svc**: `C` (float, $10^{-2}$–$10^{2}$, skala log), `kernel` (categorical: `"rbf"`, `"linear"`).
   - **logreg**: `C` (float, $10^{-2}$–$10^{2}$, skala log), `penalty` (categorical: `"l1"`, `"l2"`), `solver` ustawiony na `"liblinear"`.
3. Buduje potok: `StandardScaler` → wybrany klasyfikator.
4. Ewaluuje potok za pomocą `cross_val_score` z 5-krotną kroswalidacją i `scoring="accuracy"`.
5. Zwraca średnią accuracy.

To jest przykład *define-by-run*: przestrzeń przeszukiwania zależy od wyboru klasyfikatora w danej próbie. Tego nie można zrobić z `GridSearchCV`.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

### 3.2 Optymalizacja

Utwórz study z `direction="maximize"` (maksymalizujemy accuracy) i uruchom optymalizację z `n_trials=100`.

Wyświetl najlepsze hiperparametry i najlepszy wynik kroswalidacji.

### 3.3 Optuna Dashboard

Dotychczas korzystaliśmy z wykresów matplotlib. Optuna oferuje też interaktywny dashboard, który pozwala eksplorować wyniki optymalizacji w przeglądarce.

Aby go uruchomić, potrzebujemy zapisać study w bazie SQLite (zamiast trzymać je w pamięci):

In [ ]:
storage = "sqlite:///lab6.db"

study_db = optuna.create_study(
    study_name="breast_cancer",
    storage=storage,
    direction="maximize",
    load_if_exists=True
)
study_db.optimize(objective, n_trials=100)

W osobnym terminalu (lub w nowej komórce z `!`) uruchom dashboard:

```bash
pip install optuna-dashboard
optuna-dashboard sqlite:///lab6.db
```

Dashboard domyślnie uruchamia się na `http://localhost:8080`. Otwórz go w przeglądarce i przejrzyj dostępne wizualizacje:

- **Optimization History** — wykres wartości celu w kolejnych próbach.
- **Hyperparameter Importances** — które hiperparametry mają największy wpływ.
- **Parallel Coordinate** — wzorce w najlepszych próbach.
- **Slice Plot** — zależność wartości celu od poszczególnych hiperparametrów.

Zanotuj, który klasyfikator i jakie hiperparametry okazały się najlepsze.

### 3.4 Finalny model

Na podstawie najlepszych hiperparametrów (`study_db.best_params`) zbuduj finalny potok, wytrenuj go na pełnym zbiorze treningowym i oceń na zbiorze testowym.

## 4. Pruning na foldach kroswalidacji

W sekcji 3 użyliśmy `cross_val_score`, który zawsze wykonuje pełną kroswalidację — nawet jeśli po pierwszych dwóch foldach widać, że dana konfiguracja jest beznadziejna. `MedianPruner` pozwala przerywać takie próby wcześnie, oszczędzając czas obliczeniowy.

Mechanizm: po każdym foldzie raportujemy bieżącą średnią accuracy. Jeśli jest gorsza od mediany wyników innych prób na tym samym etapie (foldzie), próba zostaje odrzucona.

### 4.1 Funkcja celu z pruningiem

Napisz funkcję `objective_with_pruning(trial)`, która:

1. Definiuje klasyfikator i hiperparametry tak samo jak w sekcji 3.1.
2. Zamiast `cross_val_score` używa ręcznej pętli po foldach (`StratifiedKFold`, `n_splits=5`).
3. Po każdym foldzie:
   - oblicza bieżącą średnią accuracy ze wszystkich dotychczasowych foldów,
   - raportuje ją: `trial.report(running_mean, step)`,
   - sprawdza: `if trial.should_prune(): raise optuna.TrialPruned()`.
4. Po przejściu wszystkich foldów zwraca końcową średnią accuracy.

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

### 4.2 Optymalizacja z pruningiem

Utwórz study z `MedianPruner`:

In [ ]:
study_pruned = optuna.create_study(
    direction="maximize",
    pruner=optuna.pruners.MedianPruner(
        n_startup_trials=5,
        n_warmup_steps=1
    )
)
study_pruned.optimize(objective_with_pruning, n_trials=100)

Parametr `n_startup_trials=5` oznacza, że pierwsze 5 prób zawsze jest wykonywanych w całości — pruner potrzebuje danych referencyjnych. `n_warmup_steps=1` oznacza, że pruning zaczyna działać od drugiego folda (po pierwszym foldzie nie ma jeszcze dość informacji).

### 4.3 Statystyki pruningu

Sprawdź, ile prób zostało odrzuconych, a ile zakończonych w pełni:

In [ ]:
pruned = [t for t in study_pruned.trials
          if t.state == optuna.trial.TrialState.PRUNED]
complete = [t for t in study_pruned.trials
            if t.state == optuna.trial.TrialState.COMPLETE]

print(f"Zakończone: {len(complete)}, odrzucone (pruned): {len(pruned)}")
print(f"Najlepszy wynik: {study_pruned.best_value:.4f}")
print(f"Najlepsze parametry: {study_pruned.best_params}")

Jaki procent prób został odrzucony? Porównaj najlepszy wynik z wynikiem z sekcji 3.2 (bez pruningu). Czy pruning pogorszył jakość końcowego rozwiązania?

### 4.4 Wizualizacja w dashboardzie

Zapisz study z pruningiem do tej samej bazy SQLite:

In [ ]:
study_pruned_db = optuna.create_study(
    study_name="breast_cancer_pruned",
    storage=storage,
    direction="maximize",
    pruner=optuna.pruners.MedianPruner(
        n_startup_trials=5,
        n_warmup_steps=1
    ),
    load_if_exists=True
)
study_pruned_db.optimize(objective_with_pruning, n_trials=100)

Odśwież dashboard w przeglądarce. Teraz widoczne są dwa study. W study z pruningiem zwróć uwagę na:

- Próby oznaczone jako *pruned* na wykresie Optimization History.
- Intermediate Values — wykres wartości pośrednich (średnia accuracy po kolejnych foldach) dla poszczególnych prób. Odrzucone próby mają krótsze serie.

### 4.5 Finalny model

Zbuduj finalny potok na podstawie najlepszych hiperparametrów z `study_pruned`, wytrenuj na pełnym zbiorze treningowym i oceń na zbiorze testowym. Porównaj wynik z modelem z sekcji 3.4.